# PSO — irace Configuration + CNN Architecture Search

**Pipeline:**
1. Load CIFAR-10
2. Run **irace** (Optuna TPE) to find the best PSO hyperparameters (`w`, `c1`, `c2`, `pop_size`)
3. Visualise the irace study with `optuna.visualization`
4. Run the full PSO optimisation using the best parameters found
5. Train the final CNN with the best architecture found

> **Two-level optimisation:**  
> Optuna (outer) → finds best PSO parameters  
> PSO (inner) → finds best CNN architecture

## 1. Setup

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import sys
sys.path.append('..')   # makes Utils importable if running from a subdirectory

import optuna
import optuna.visualization as vis
import pandas as pd
import matplotlib.pyplot as plt

from Utils.datasets  import getDataset
from Utils.irace     import configureOptimizationAlgorithmIRACE
from Utils.optimizer import runOptimization, trainBest, plotOptimizationHistory

optuna.logging.set_verbosity(optuna.logging.WARNING)
os.makedirs('results/irace',         exist_ok=True)
os.makedirs('results/pso_final',     exist_ok=True)

print('Setup complete.')

## 2. Configuration

In [ ]:
# ── irace budget (reduced for speed) ──────────────────────────────────────────
N_TRIALS       = 20      # number of PSO configurations to race
IRACE_N_ITER   = 5       # PSO iterations per race
POP_SIZE_RANGE = (10,40) # Optuna searches pop_size in this range
IRACE_EPOCHS   = 3       # CNN eval epochs per race evaluation
# → evaluations per trial ≈ (IRACE_N_ITER + 1) × mid(POP_SIZE_RANGE) = ~150

# ── Full PSO run (after irace) ────────────────────────────────────────────────
FINAL_N_ITER   = 20      # PSO iterations for the main run
FINAL_EPOCHS   = 5       # CNN eval epochs for the main run

# ── Final CNN training ────────────────────────────────────────────────────────
TRAIN_EPOCHS   = 50      # max epochs (EarlyStopping will stop earlier)

SEED = 42

## 3. Load CIFAR-10

In [ ]:
data = getDataset('cifar10')

print(f"  x_train : {data['x_train'].shape}")
print(f"  x_test  : {data['x_test'].shape}")
print(f"  classes : {data['class_names']}")

## 4. irace — Find best PSO hyperparameters

Optuna (TPE multivariate) races `N_TRIALS` PSO configurations.  
Each trial runs PSO with reduced budget and reports `val_accuracy` back to Optuna.  
The parameters searched are: `w`, `c1`, `c2`, `pop_size`.

In [ ]:
best_params, study = configureOptimizationAlgorithmIRACE(
    algorithm      = 'PSO',
    data           = data,
    n_trials       = N_TRIALS,
    n_iter         = IRACE_N_ITER,
    pop_size_range = POP_SIZE_RANGE,
    eval_epochs    = IRACE_EPOCHS,
    results_dir    = 'results/irace',
    seed           = SEED,
)

print(f"\nBest PSO parameters found:")
for k, v in best_params.items():
    print(f"  {k:<12} {v}")

## 5. Optuna Visualisations

Interactive Plotly charts showing the irace study results.

In [ ]:
# 5a. Optimisation history — val_accuracy per trial + running best
fig = vis.plot_optimization_history(study)
fig.update_layout(title='irace PSO — Optimisation History')
fig.show()

In [ ]:
# 5b. Parameter importances — which PSO parameter had the most impact
fig = vis.plot_param_importances(study)
fig.update_layout(title='irace PSO — Parameter Importances')
fig.show()

In [ ]:
# 5c. Contour plot — interaction between the two most important parameters
fig = vis.plot_contour(study, params=['w', 'c1'])
fig.update_layout(title='irace PSO — Contour: w vs c1')
fig.show()

In [ ]:
# 5d. Parallel coordinates — all parameters across all trials
fig = vis.plot_parallel_coordinate(study)
fig.update_layout(title='irace PSO — Parallel Coordinates')
fig.show()

In [ ]:
# 5e. Slice plot — effect of each parameter individually on val_accuracy
fig = vis.plot_slice(study)
fig.update_layout(title='irace PSO — Slice Plot')
fig.show()

In [ ]:
# 5f. Summary table of all trials
df = study.trials_dataframe()
df = df[['number','value'] + [c for c in df.columns if c.startswith('params_')]].copy()
df.columns = ['trial','val_accuracy'] + [c.replace('params_','') for c in df.columns if c.startswith('params_')]
df = df.sort_values('val_accuracy', ascending=False).reset_index(drop=True)
print(f"Top 10 trials:")
df.head(10)

## 6. Full PSO Optimisation with best parameters

Now we use the parameters found by irace to run the full PSO search over the 11D CNN architecture space.

In [ ]:
# Separate pop_size from algo_params (pop_size is a runOptimization argument)
pso_params = dict(best_params)           # copy to avoid modifying best_params
pop_size   = pso_params.pop('pop_size')  # extract pop_size
# pso_params now contains only: w, c1, c2

print(f"Running PSO with:")
print(f"  pop_size   = {pop_size}")
print(f"  n_iter     = {FINAL_N_ITER}")
print(f"  eval_epochs= {FINAL_EPOCHS}")
for k, v in pso_params.items():
    print(f"  {k:<12} = {v:.4f}")
print(f"  total evals= {(FINAL_N_ITER + 1) * pop_size}")

In [ ]:
best_config, run_results = runOptimization(
    algorithm   = 'PSO',
    data        = data,
    eval_epochs = FINAL_EPOCHS,
    n_iter      = FINAL_N_ITER,
    pop_size    = pop_size,
    algo_params = pso_params,
    results_dir = 'results/pso_final',
    seed        = SEED,
)

## 7. PSO Optimisation Results

In [ ]:
plotOptimizationHistory(run_results, save_path='results/pso_final/pso_history.png')

In [ ]:
print("\nBest CNN architecture found by PSO:")
print("-" * 40)
for k, v in best_config.items():
    print(f"  {k:<16} {v}")
print(f"\n  Best val_accuracy (search): {run_results['best']['val_accuracy']:.4f}")

## 8. Final CNN Training

Train the best architecture found by PSO for the full number of epochs.  
EarlyStopping (patience=5) will stop before `TRAIN_EPOCHS` if val_accuracy plateaus.

In [ ]:
results = trainBest(
    best_config             = best_config,
    data                    = data,
    epochs                  = TRAIN_EPOCHS,
    early_stopping_patience = 5,
    results_dir             = 'results/pso_final',
)

## 9. Final Results

In [ ]:
history = results['history']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Final CNN Training — PSO Optimised Architecture', fontsize=13, fontweight='bold')

epochs_range = range(1, len(history['accuracy']) + 1)

ax1.plot(epochs_range, history['accuracy'],     label='Train', color='steelblue')
ax1.plot(epochs_range, history['val_accuracy'], label='Val',   color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['loss'],     label='Train', color='steelblue')
ax2.plot(epochs_range, history['val_loss'], label='Val',   color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/pso_final/final_training.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal test accuracy: {results['metrics']['accuracy']:.4f}  "
      f"({results['metrics']['accuracy']*100:.2f}%)")